In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Identify the project root directory
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ChargingRecords_Cleaned.csv"
)

# Load cleaned data
df = pd.read_csv(DATA_PATH)

# Parse datetime column
df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)

df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)

print("CLEANED DATA LOADED SUCCESSFULLY")
print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())

display(df.head())

CLEANED DATA LOADED SUCCESSFULLY
Dataset shape: (72826, 21)
Missing values: 0


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,...,Duration,Demand,StartHour,StartMonth,StartWeekday,StartDate,DurationFromTimestamps,DurationDifference,DurationMismatch,DurationQuality
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,...,185,20.36,20,9,Thursday,2022-09-15,185.183333,-0.183333,0,Match
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,...,90,10.19,20,9,Wednesday,2022-09-14,89.983333,0.016667,0,Match
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,...,60,6.78,18,9,Wednesday,2022-09-14,59.983333,0.016667,0,Match
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,...,344,37.65,18,9,Thursday,2022-09-29,343.850000,0.150000,0,Match
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:15,...,300,33.81,19,9,Sunday,2022-09-25,299.983333,0.016667,0,Match


In [2]:

# Create time-based features
df["StartHour"] = df["StartDatetime"].dt.hour
df["StartMonth"] = df["StartDatetime"].dt.month
df["StartDayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["StartDayOfMonth"] = df["StartDatetime"].dt.day

# Monday-Friday = Weekday; Saturday-Sunday = Weekend
df["IsWeekend"] = (
    df["StartDayOfWeek"] >= 5
).astype(int)

# Create classification target using median demand
demand_threshold = df["Demand"].median()

df["DemandClass"] = np.where(
    df["Demand"] >= demand_threshold,
    "High",
    "Normal"
)

print("FEATURE ENGINEERING COMPLETED")
print(f"Median demand threshold: {demand_threshold:.2f}")

print("\nDemand class distribution:")
display(
    df["DemandClass"]
    .value_counts()
    .rename_axis("DemandClass")
    .to_frame("RecordCount")
)

print("\nEngineered features:")
display(
    df[
        [
            "Demand",
            "StartHour",
            "StartMonth",
            "StartDayOfWeek",
            "StartDayOfMonth",
            "IsWeekend",
            "DemandClass"
        ]
    ].head()
)

FEATURE ENGINEERING COMPLETED
Median demand threshold: 14.10

Demand class distribution:


,RecordCount
DemandClass,
High,36439
Normal,36387



Engineered features:


,Demand,StartHour,StartMonth,StartDayOfWeek,StartDayOfMonth,IsWeekend,DemandClass
0,20.36,20,9,3,15,0,High
1,10.19,20,9,2,14,0,Normal
2,6.78,18,9,2,14,0,Normal
3,37.65,18,9,3,29,0,High
4,33.81,19,9,6,25,1,High


In [3]:

print("FEATURE VALIDATION")
print("Total records:", len(df))
print("Missing start timestamps:", df["StartDatetime"].isna().sum())
print("Missing demand values:", df["Demand"].isna().sum())

print("\nHour range:")
print(df["StartHour"].min(), "to", df["StartHour"].max())

print("\nMonth range:")
print(df["StartMonth"].min(), "to", df["StartMonth"].max())

print("\nDemand class labels:")
print(sorted(df["DemandClass"].unique()))

assert df["Demand"].notna().all()
assert df["StartHour"].between(0, 23).all()
assert df["StartMonth"].between(1, 12).all()
assert set(df["DemandClass"].unique()) == {"High", "Normal"}

print("\nALL FEATURE VALIDATION CHECKS PASSED")

FEATURE VALIDATION
Total records: 72826
Missing start timestamps: 0
Missing demand values: 0

Hour range:
0 to 23

Month range:
1 to 12

Demand class labels:
['High', 'Normal']

ALL FEATURE VALIDATION CHECKS PASSED


In [6]:


import numpy as np
import pandas as pd
import scipy
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

NumPy: 2.5.4
Pandas: 3.0.6
SciPy: 1.18.1
Scikit-learn: 1.9.1


In [10]:

from sklearn.model_selection import train_test_split

# 1. Select features available at prediction time
feature_columns = [
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartHour",
    "StartMonth",
    "StartDayOfWeek",
    "StartDayOfMonth",
    "IsWeekend",
]

# 2. Prepare features and targets
X = df[feature_columns].copy()
y_regression = df["Demand"].copy()
y_classification = df["DemandClass"].copy()

# 3. Encode categorical Location values
X = pd.get_dummies(
    X,
    columns=["Location"],
    drop_first=False,
    dtype=int
)

# 4. Split regression data (80% train, 20% test)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_regression,
    test_size=0.20,
    random_state=42
)

# 5. Split classification data (preserve class proportions)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

# 6. Validate the prepared datasets
print("MODEL INPUT PREPARATION COMPLETED")
print("-" * 45)
print("Total records:", len(X))
print("Features after encoding:", X.shape[1])
print("Regression training shape:", X_train_reg.shape)
print("Regression testing shape:", X_test_reg.shape)
print("Classification training shape:", X_train_clf.shape)
print("Classification testing shape:", X_test_clf.shape)

print("\nClassification training distribution:")
print(y_train_clf.value_counts())

print("\nMissing values in features:", X.isna().sum().sum())
print("Missing values in regression target:", y_regression.isna().sum())
print("Missing values in classification target:", y_classification.isna().sum())

MODEL INPUT PREPARATION COMPLETED
---------------------------------------------
Total records: 72826
Features after encoding: 21
Regression training shape: (58260, 21)
Regression testing shape: (14566, 21)
Classification training shape: (58260, 21)
Classification testing shape: (14566, 21)

Classification training distribution:
DemandClass
High      29151
Normal    29109
Name: count, dtype: int64

Missing values in features: 0
Missing values in regression target: 0
Missing values in classification target: 0


In [8]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find the project folder
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load the cleaned dataset
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ChargingRecords_Cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

# Convert timestamp columns
df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)
df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)

print("DATA LOADING COMPLETED")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

DATA LOADING COMPLETED
Dataset shape: (72826, 21)
Columns: ['UserID', 'ChargerID', 'ChargerCompany', 'Location', 'ChargerType', 'StartDay', 'StartTime', 'EndDay', 'EndTime', 'StartDatetime', 'EndDatetime', 'Duration', 'Demand', 'StartHour', 'StartMonth', 'StartWeekday', 'StartDate', 'DurationFromTimestamps', 'DurationDifference', 'DurationMismatch', 'DurationQuality']


In [9]:

df["StartHour"] = df["StartDatetime"].dt.hour
df["StartMonth"] = df["StartDatetime"].dt.month
df["StartDayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["StartDayOfMonth"] = df["StartDatetime"].dt.day
df["IsWeekend"] = (df["StartDayOfWeek"] >= 5).astype(int)

demand_threshold = df["Demand"].median()

df["DemandClass"] = np.where(
    df["Demand"] >= demand_threshold,
    "High",
    "Normal"
)

print("FEATURE ENGINEERING COMPLETED")
print("Demand threshold:", demand_threshold)
print(df["DemandClass"].value_counts())

FEATURE ENGINEERING COMPLETED
Demand threshold: 14.1
DemandClass
High      36439
Normal    36387
Name: count, dtype: int64


In [11]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# Train the baseline regression model
linear_model = LinearRegression()
linear_model.fit(X_train_reg, y_train_reg)

# Predict demand on unseen test data
y_pred_reg = linear_model.predict(X_test_reg)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

# Display results
regression_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2]
})

print("LINEAR REGRESSION — BASELINE RESULTS")
print(regression_results.to_string(index=False))

LINEAR REGRESSION — BASELINE RESULTS
Metric      Value
   MAE   9.990136
   MSE 171.138301
  RMSE  13.081984
    R²   0.050277


In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project folder
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load the cleaned dataset
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ChargingRecords_Cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

# Convert timestamps
df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)
df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print(df.head())

Dataset loaded successfully!
Dataset shape: (72826, 21)
   UserID  ChargerID  ChargerCompany Location  ChargerType    StartDay  \
0       0          1               1    hotel            0  2022-09-15   
1       0          1               1    hotel            0  2022-09-14   
2       0          1               1    hotel            0  2022-09-14   
3       0          1               1    hotel            0  2022-09-29   
4       0          1               1    hotel            0  2022-09-25   

  StartTime      EndDay   EndTime       StartDatetime  ... Duration  Demand  \
0  20:54:02  2022-09-15  23:59:13 2022-09-15 20:54:02  ...      185   20.36   
1  20:01:05  2022-09-14  21:31:04 2022-09-14 20:01:05  ...       90   10.19   
2  18:54:30  2022-09-14  19:54:29 2022-09-14 18:54:30  ...       60    6.78   
3  18:32:51  2022-09-30   0:16:42 2022-09-29 18:32:51  ...      344   37.65   
4  19:30:15  2022-09-26   0:30:14 2022-09-25 19:30:15  ...      300   33.81   

   StartHour  StartMonth

In [4]:

df["StartHour"] = df["StartDatetime"].dt.hour
df["StartMonth"] = df["StartDatetime"].dt.month
df["StartDayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["StartDayOfMonth"] = df["StartDatetime"].dt.day
df["IsWeekend"] = (df["StartDayOfWeek"] >= 5).astype(int)

demand_threshold = df["Demand"].median()

df["DemandClass"] = np.where(
    df["Demand"] >= demand_threshold,
    "High",
    "Normal"
)

print("Feature engineering restored!")
print("Demand threshold:", demand_threshold)
print(df[["Demand", "StartHour", "StartMonth", "IsWeekend", "DemandClass"]].head())

Feature engineering restored!
Demand threshold: 14.1
   Demand  StartHour  StartMonth  IsWeekend DemandClass
0   20.36         20           9          0        High
1   10.19         20           9          0      Normal
2    6.78         18           9          0      Normal
3   37.65         18           9          0        High
4   33.81         19           9          1        High


In [5]:

from sklearn.model_selection import train_test_split

feature_columns = [
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartHour",
    "StartMonth",
    "StartDayOfWeek",
    "StartDayOfMonth",
    "IsWeekend",
]

X = df[feature_columns].copy()
y_regression = df["Demand"].copy()
y_classification = df["DemandClass"].copy()

X = pd.get_dummies(
    X,
    columns=["Location"],
    drop_first=False,
    dtype=int
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_regression,
    test_size=0.20,
    random_state=42
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

print("Model inputs restored successfully!")
print("Training features:", X_train_reg.shape)
print("Testing features:", X_test_reg.shape)

Model inputs restored successfully!
Training features: (58260, 21)
Testing features: (14566, 21)


In [6]:

# ==========================================
# EV CHARGING DEMAND - RANDOM FOREST REGRESSION
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# STEP 1: LOAD DATASET
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "ChargingRecords_Cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)
df["EndDatetime"] = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)

print("Dataset loaded:", df.shape)

# STEP 2: FEATURE ENGINEERING
df["StartHour"] = df["StartDatetime"].dt.hour
df["StartMonth"] = df["StartDatetime"].dt.month
df["StartDayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["StartDayOfMonth"] = df["StartDatetime"].dt.day
df["IsWeekend"] = (df["StartDayOfWeek"] >= 5).astype(int)

demand_threshold = df["Demand"].median()

df["DemandClass"] = np.where(
    df["Demand"] >= demand_threshold,
    "High",
    "Normal"
)

print("Demand threshold:", demand_threshold)

# STEP 3: PREPARE MODEL INPUTS
feature_columns = [
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartHour",
    "StartMonth",
    "StartDayOfWeek",
    "StartDayOfMonth",
    "IsWeekend"
]

X = df[feature_columns].copy()
y_regression = df["Demand"].copy()

X = pd.get_dummies(
    X,
    columns=["Location"],
    drop_first=False,
    dtype=int
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = (
    train_test_split(
        X,
        y_regression,
        test_size=0.20,
        random_state=42
    )
)

print("Training shape:", X_train_reg.shape)
print("Testing shape:", X_test_reg.shape)

# STEP 4: TRAIN LINEAR REGRESSION BASELINE
linear_model = LinearRegression()
linear_model.fit(X_train_reg, y_train_reg)

y_pred_linear = linear_model.predict(X_test_reg)

# STEP 5: TRAIN RANDOM FOREST REGRESSION
rf_regression_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_regression_model.fit(X_train_reg, y_train_reg)

y_pred_rf = rf_regression_model.predict(X_test_reg)

# STEP 6: EVALUATE BOTH MODELS
def evaluate_regression(model_name, actual, predicted):
    mse = mean_squared_error(actual, predicted)

    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted)
    }

regression_comparison = pd.DataFrame([
    evaluate_regression(
        "Linear Regression", y_test_reg, y_pred_linear
    ),
    evaluate_regression(
        "Random Forest", y_test_reg, y_pred_rf
    )
])

print("\nREGRESSION MODEL COMPARISON")
print(regression_comparison.round(4).to_string(index=False))

# STEP 7: FEATURE IMPORTANCE
feature_importance = pd.DataFrame({
    "Feature": X_train_reg.columns,
    "Importance": rf_regression_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nTOP 10 IMPORTANT FEATURES")
print(feature_importance.head(10).to_string(index=False))

print("\nRANDOM FOREST REGRESSION COMPLETED!")

Dataset loaded: (72826, 21)
Demand threshold: 14.1
Training shape: (58260, 21)
Testing shape: (14566, 21)

REGRESSION MODEL COMPARISON
            Model    MAE      MSE    RMSE     R2
Linear Regression 9.9901 171.1383 13.0820 0.0503
    Random Forest 9.4409 157.6938 12.5576 0.1249

TOP 10 IMPORTANT FEATURES
             Feature  Importance
           StartHour    0.297508
     StartDayOfMonth    0.207501
          StartMonth    0.155134
      StartDayOfWeek    0.101641
         ChargerType    0.038479
      ChargerCompany    0.028611
     Location_market    0.023299
    Location_camping    0.021120
      Location_hotel    0.018151
Location_public area    0.015055

RANDOM FOREST REGRESSION COMPLETED!


In [7]:

# ==========================================
# HIGH / NORMAL DEMAND CLASSIFICATION
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# STEP 1: CREATE DEMAND CLASSES
demand_threshold = df["Demand"].median()

df["DemandClass"] = np.where(
    df["Demand"] >= demand_threshold,
    "High",
    "Normal"
)

print("Demand threshold:", demand_threshold)
print("\nClass distribution:")
print(df["DemandClass"].value_counts())

# STEP 2: PREPARE FEATURES
feature_columns = [
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartHour",
    "StartMonth",
    "StartDayOfWeek",
    "StartDayOfMonth",
    "IsWeekend"
]

X_clf = df[feature_columns].copy()
y_clf = df["DemandClass"].copy()

X_clf = pd.get_dummies(
    X_clf,
    columns=["Location"],
    drop_first=False,
    dtype=int
)

# STEP 3: SPLIT DATA
X_train_clf, X_test_clf, y_train_clf, y_test_clf = (
    train_test_split(
        X_clf,
        y_clf,
        test_size=0.20,
        random_state=42,
        stratify=y_clf
    )
)

# STEP 4: TRAIN RANDOM FOREST CLASSIFIER
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_classifier.fit(X_train_clf, y_train_clf)

# STEP 5: PREDICT
y_pred_clf = rf_classifier.predict(X_test_clf)
y_prob_clf = rf_classifier.predict_proba(X_test_clf)

# Find the probability column for High demand
high_index = list(rf_classifier.classes_).index("High")
y_prob_high = y_prob_clf[:, high_index]

# STEP 6: EVALUATE
results = {
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Value": [
        accuracy_score(y_test_clf, y_pred_clf),
        precision_score(
            y_test_clf, y_pred_clf,
            pos_label="High", zero_division=0
        ),
        recall_score(
            y_test_clf, y_pred_clf,
            pos_label="High", zero_division=0
        ),
        f1_score(
            y_test_clf, y_pred_clf,
            pos_label="High", zero_division=0
        ),
        roc_auc_score(
            (y_test_clf == "High").astype(int),
            y_prob_high
        )
    ]
}

classification_results = pd.DataFrame(results)

print("\nCLASSIFICATION MODEL RESULTS")
print(classification_results.round(4).to_string(index=False))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test_clf,
    y_pred_clf,
    zero_division=0
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(
    y_test_clf,
    y_pred_clf,
    labels=["High", "Normal"]
))

print("\nCLASSIFICATION COMPLETED!")

Demand threshold: 14.1

Class distribution:
DemandClass
High      36439
Normal    36387
Name: count, dtype: int64

CLASSIFICATION MODEL RESULTS
   Metric  Value
 Accuracy 0.6541
Precision 0.6536
   Recall 0.6567
 F1 Score 0.6552
  ROC-AUC 0.7094

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        High       0.65      0.66      0.66      7288
      Normal       0.65      0.65      0.65      7278

    accuracy                           0.65     14566
   macro avg       0.65      0.65      0.65     14566
weighted avg       0.65      0.65      0.65     14566


CONFUSION MATRIX
[[4786 2502]
 [2536 4742]]

CLASSIFICATION COMPLETED!
